
## Training AI agent tool behavior 

* ollama run mistral:7b


In [1]:

## !pip install scikit-learn


In [2]:


from crewai import Agent
from langchain.tools import Tool

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



In [3]:

import os
from crewai_tools import SerperDevTool, BrowserbaseLoadTool, EXASearchTool
from crewai import Agent
from crewai import Task
from crewai import LLM

from langchain_community.llms import Ollama


from crewai import Crew, Process


In [4]:

import pandas as pd
from crewai import Agent, Task, Crew

from langchain.tools import tool



## Read data


In [5]:

import pandas as pd
df = pd.read_csv("sales.csv")

df



,Date,Revenue,Expenses,Customers
0,2024-01-31,12000,8000,234
1,2024-02-29,10000,8500,201
2,2024-03-31,9300,8700,198
3,2024-04-30,11500,8100,225
4,2024-05-31,9800,7900,210
5,2024-06-30,10400,8000,215



## Define Tools


In [6]:

@tool
def summarize_csv( x: str ):          ##  -> str:
    """Summarizes the dataframe by showing the first few rows and statistical metrics."""
    summary = df.describe(include='all').to_string()
    preview = df.head().to_string()
    return f"Data Preview:\n{preview}\n\nSummary Statistics:\n{summary}"



In [7]:


def calculator_tool(x: str):
    try:
        return str(  eval(x)  )
    except:
        return "Error in calculation"


In [8]:

def search_tool(x: str):
    return f"Searching for: {x}"



## Log Trajectory data


In [9]:


logs = []

def logging_tool(tool_name, func):
    
    def wrapper(x):
        logs.append(  (x, tool_name)  )
        return func(x)
        
    return wrapper



In [10]:

tools = [
    Tool(
        name="Calculator",
        func=logging_tool("Calculator", calculator_tool),
        description="Use for math operations"
    ),
    Tool(
        name="Search",
        func=logging_tool("Search", search_tool),
        description="Use for general search based knowledge"
    ),
    Tool(
        name="TableSummarizer",
        func=logging_tool("TableSummarizer", summarize_csv),
        description="Use for CSV summarization"
    )
]



In [11]:

print("Collected logs:")
print(logs)


Collected logs:
[]



## Run agent behavioral trajectory


In [12]:


queries = [
    "2 + 2",
    "summarize the CSV",
    "10 * 5",
    "100 / 4",
    "Who is Einstein?",
    "capital of France",
    "latest news",
    "3 * 9",
    "square root of 16"
]



In [13]:


print("\n--- Collecting data from agent ---\n")

for q in queries:
    try: 
        
        topic = q

        agent = Agent(
            role='Assistant',
            goal=f"This is your task: {topic} ",
            backstory="""You are a very smart agent and you choose tools wisely.""",
            verbose=False,
            tools=tools,
            llm=LLM( model="ollama/mistral:7b", base_url="http://localhost:11434" )
        )

        task = Task(
            description=f"This is your task: {topic} ",
            expected_output="Successfully resolving the task.",
            agent=agent,
            ## tools=[search_tool],
           async_execution=False
        )

        crew = Crew(
            agents=[agent],
            tasks =[ task],
            process=Process.sequential,
            verbose=True
        )

        result = crew.kickoff(  inputs={"topic": topic }  ) 
        
        print(  result  )
    except Exception as e:
        print(e)



--- Collecting data from agent ---

# Agent: Assistant
## Task: This is your task: 2 + 2 


# Agent: Assistant
## Using tool: Calculator
## Tool Input: 
"{\"x\": 2}"
## Tool Output: 
Error in calculation


# Agent: Assistant
## Thought: Thought: The 'Calculator' tool requires numerical input. Since I only have the number 2, let me calculate 2 + 2 using that.
## Using tool: Calculator
## Tool Input: 
"{\"x\": 2}"
## Tool Output: 
I tried reusing the same input, I must stop using this action input. I'll try something else instead.




# Agent: Assistant
## Thought: Thought: Apologies for the confusion earlier. Now I will provide the correct format.
## Using tool: Calculator
## Tool Input: 
"{\"x\": 2, \"y\": 2}"
## Tool Output: 
Error in calculation


You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:

Tool Name: Calculator
Tool Arguments: {'x': {'description': '', 'type': 'Any'}}
Tool Description: Use for math operations
Tool Name: Sea

2026-04-19 15:31:22,126 - 8644006464 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed




# Agent: Assistant
## Final Answer: 
To add two numbers (for example 2 and 3) in the Calculator tool, the input should be {'x': {'num1': 2, 'num2': 3}}. The output would then be the sum of these two numbers.


To add two numbers (for example 2 and 3) in the Calculator tool, the input should be {'x': {'num1': 2, 'num2': 3}}. The output would then be the sum of these two numbers.
# Agent: Assistant
## Task: This is your task: summarize the CSV 


/var/folders/q4/zdsjyw0d297_fn6_fh5n7g9h0000gn/T/ipykernel_33918/1590334649.py:7: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return func(x)




# Agent: Assistant
## Using tool: TableSummarizer
## Tool Input: 
"{\"x\": \"summarize the CSV\"}"
## Tool Output: 
Data Preview:
         Date  Revenue  Expenses  Customers
0  2024-01-31    12000      8000        234
1  2024-02-29    10000      8500        201
2  2024-03-31     9300      8700        198
3  2024-04-30    11500      8100        225
4  2024-05-31     9800      7900        210

Summary Statistics:
              Date       Revenue    Expenses   Customers
count            6      6.000000     6.00000    6.000000
unique           6           NaN         NaN         NaN
top     2024-01-31           NaN         NaN         NaN
freq             1           NaN         NaN         NaN
mean           NaN  10500.000000  8200.00000  213.833333
std            NaN   1043.072385   322.49031   13.876839
min            NaN   9300.000000  7900.00000  198.000000
25%            NaN   9850.000000  8000.00000  203.250000
50%            NaN  10200.000000  8050.00000  212.500000
75%          

...



# Agent: Assistant
## Final Answer: 
The capital of France is Paris.


The capital of France is Paris.
# Agent: Assistant
## Task: This is your task: latest news 


# Agent: Assistant
## Using tool: Search
## Tool Input: 
"{\"x\": \"latest news\"}"
## Tool Output: 
Searching for: latest news


# Agent: Assistant
## Using tool: Search
## Tool Input: 
"{\"x\": \"latest news\"}"
## Tool Output: 
I tried reusing the same input, I must stop using this action input. I'll try something else instead.




# Agent: Assistant
## Using tool: Search
## Tool Input: 
"{\"x\": \"reliable news websites\"}"
## Tool Output: 
Searching for: reliable news websites


You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:

Tool Name: Calculator
Tool Arguments: {'x': {'description': '', 'type': 'Any'}}
Tool Description: Use for math operations
Tool Name: Search
Tool Arguments: {'x': {'description': '', 'type': 'Any'}}
Tool Description: Use for general search ba

2026-04-19 15:38:59,803 - 8644006464 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed




# Agent: Assistant
## Final Answer: 
To get the latest news from CNN, follow these steps:
1. Find a web scraping API that can fetch news articles from CNN.
2. Use this API to access the CNN News API documentation.
3. Follow the instructions in the documentation to fetch the latest news article.


To get the latest news from CNN, follow these steps:
1. Find a web scraping API that can fetch news articles from CNN.
2. Use this API to access the CNN News API documentation.
3. Follow the instructions in the documentation to fetch the latest news article.
# Agent: Assistant
## Task: This is your task: 3 * 9 


# Agent: Assistant
## Using tool: Calculator
## Tool Input: 
"{\"x\": 3}"
## Tool Output: 
Error in calculation


# Agent: Assistant
## Thought: Thought: The calculator tool is not able to perform multiplication directly. I need to use the correct syntax for multiplication in the calculator tool.
## Using tool: Calculator
## Tool Input: 
"{\"x\": \"3*9\"}"
## Tool Output: 
27


2026-04-19 15:39:12,877 - 8644006464 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed




# Agent: Assistant
## Final Answer: 
27


27
# Agent: Assistant
## Task: This is your task: square root of 16 


# Agent: Assistant
## Using tool: Calculator
## Tool Input: 
"{\"x\": 16}"
## Tool Output: 
Error in calculation


# Agent: Assistant
## Thought: Thought: It seems there's an error with the input. The square root of a number should be a positive value, not an error. I will correct it and try again.
## Using tool: Calculator
## Tool Input: 
"{\"x\": 144}"
## Tool Output: 
Error in calculation


# Agent: Assistant
## Thought: Thought: It looks like there was another mistake with the input. Instead of squaring the number, I should take its square root. Let me correct the error and try again.
## Using tool: Calculator
## Tool Input: 
"{\"x\": 16.0}"
## Tool Output: 
Error in calculation


You ONLY have access to the following tools, and should NEVER make up tools that are not listed here:

Tool Name: Calculator
Tool Arguments: {'x': {'description': '', 'type': 'Any'}}
Tool Des

In [14]:

print("Collected logs:")
print(logs)


Collected logs:
[(2, 'Calculator'), (2, 'Calculator'), (2, 'Calculator'), (2, 'Calculator'), ('Calculator tool documentation', 'Search'), ('Calculator tool usage examples', 'Search'), ('Proper usage of Calculator tool', 'Search'), ('Example of adding two numbers with Calculator tool', 'Search'), ({'operation': 'add', 'num1': 2, 'num2': 3}, 'Calculator'), ('summarize the CSV', 'TableSummarizer'), (10, 'Calculator'), ('int(10) * int(5)', 'Calculator'), (100, 'Calculator'), (1, 'Calculator'), ('1 divided by 4', 'Search'), ('Equivalent decimal of 1/4', 'Search'), ('int(100 * 4)', 'Calculator'), ('How to calculate 100 divided by 4 using multiplication', 'Search'), ('Equivalent decimal of 2/5', 'Search'), ('Einstein biography', 'Search'), ('Ein Albert Einstein biography', 'Search'), ('capital of France', 'Search'), ('latest news', 'Search'), ('reliable news websites', 'Search'), ('latest headlines CNN', 'Search'), ('web scraping tools', 'Search'), ('web scraping APIs', 'Search'), ('CNN news 

In [15]:


'''

logs.append(('2+2', 'Calculator'))
logs.append(('7-3', 'Calculator'))

print(logs)

'''


"\n\nlogs.append(('2+2', 'Calculator'))\nlogs.append(('7-3', 'Calculator'))\n\nprint(logs)\n\n"


## TRAIN Tool Model based on trajectories (prompt -> TOOL)


In [16]:


texts  = [x[0] for x in logs]
labels = [x[1] for x in logs]


In [17]:

texts


[2,
 2,
 2,
 2,
 'Calculator tool documentation',
 'Calculator tool usage examples',
 'Proper usage of Calculator tool',
 'Example of adding two numbers with Calculator tool',
 {'operation': 'add', 'num1': 2, 'num2': 3},
 'summarize the CSV',
 10,
 'int(10) * int(5)',
 100,
 1,
 '1 divided by 4',
 'Equivalent decimal of 1/4',
 'int(100 * 4)',
 'How to calculate 100 divided by 4 using multiplication',
 'Equivalent decimal of 2/5',
 'Einstein biography',
 'Ein Albert Einstein biography',
 'capital of France',
 'latest news',
 'reliable news websites',
 'latest headlines CNN',
 'web scraping tools',
 'web scraping APIs',
 'CNN news API',
 'CNN news API documentation',
 'Fetching latest news using CNN News API',
 3,
 '3*9',
 16,
 144,
 16.0,
 'square root of 16',
 'Square root of sixteen',
 4.0,
 'mathematical constant square root of 16',
 4,
 'The mathematical constant square root of 16 from Wolfram Alpha',
 'The mathematical constant square root of 16 from the previous search',
 ['The ma

In [18]:

labels


['Calculator',
 'Calculator',
 'Calculator',
 'Calculator',
 'Search',
 'Search',
 'Search',
 'Search',
 'Calculator',
 'TableSummarizer',
 'Calculator',
 'Calculator',
 'Calculator',
 'Calculator',
 'Search',
 'Search',
 'Calculator',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Search',
 'Calculator',
 'Calculator',
 'Calculator',
 'Calculator',
 'Calculator',
 'Search',
 'Search',
 'Calculator',
 'Search',
 'Calculator',
 'Search',
 'Search',
 'Search']

In [20]:

texts2 = [ str(text) for text in texts]
texts  = texts2


In [21]:

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

dtm_pd = pd.DataFrame( X.toarray(), index=texts, columns=vectorizer.get_feature_names_out()  ).head(10)
dtm_pd


,10,100,144,16,add,adding,albert,alpha,api,apis,...,to,tool,tools,two,usage,using,web,websites,with,wolfram
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Calculator tool documentation,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.546546,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Calculator tool usage examples,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.446846,0.0,0.000000,0.518745,0.0,0.0,0.0,0.000000,0.0
Proper usage of Calculator tool,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.425137,0.0,0.000000,0.493543,0.0,0.0,0.0,0.000000,0.0
Example of adding two numbers with Calculator tool,0.0,0.0,0.0,0.0,0.0,0.391621,0.0,0.0,0.0,0.0,...,0.0,0.303908,0.0,0.391621,0.000000,0.0,0.0,0.0,0.391621,0.0
"{'operation': 'add', 'num1': 2, 'num2': 3}",0.0,0.0,0.0,0.0,0.5,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
summarize the CSV,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0


In [22]:



model = LogisticRegression()
model.fit(X, labels)

print("\n--- Model trained ---\n")



--- Model trained ---



In [23]:

# -------------------------------
# 6. TRAINED POLICY (PREDICT TOOL)
# -------------------------------
def predict_tool(query):
    X = vectorizer.transform([query])
    return model.predict(X)[0]


In [24]:

# -------------------------------
# 7. FINAL AGENT (USES TRAINED POLICY)
# -------------------------------
def run_agent(query):
    chosen_tool = predict_tool(query)

    print(f"\nQuery: {query}")
    print(f"Chosen Tool (trained): {chosen_tool}")

    if chosen_tool == "Calculator":
        result = calculator_tool(query)
    else:
        result = search_tool(query)

    print(f"Result: {result}")
    return result


In [25]:

# -------------------------------
# 8. TEST
# -------------------------------
print("\n--- Testing trained agent ---")

run_agent("5 * 6")
run_agent("Who invented electricity?")
run_agent("12 + 45")
run_agent("population of Japan")




--- Testing trained agent ---

Query: 5 * 6
Chosen Tool (trained): Calculator
Result: 30

Query: Who invented electricity?
Chosen Tool (trained): Calculator
Result: Error in calculation

Query: 12 + 45
Chosen Tool (trained): Calculator
Result: 57

Query: population of Japan
Chosen Tool (trained): Search
Result: Searching for: population of Japan


'Searching for: population of Japan'